*0.3 Classical NLP*

# Stopwords

**The situation.** The "top issues" word cloud is dominated by "the", "I", "my", "to", "is". They are half of all words in every ticket and carry nothing about the issue. Meanwhile a sentiment model trained on tickets with those words removed cannot tell "not working" from "working" — because "not" was on the list.

**Stopwords.** Very common words with little meaning on their own. Removing them cuts noise for counting and keyword search. Removing them blindly also destroys meaning: "not", "no", "never" are on most default lists.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import spacy
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

nlp = spacy.load("en_core_web_sm")
ticket = "I was not able to get my refund and the app is not working"

print(
    "sklearn list has",
    len(ENGLISH_STOP_WORDS),
    "words | spaCy list has",
    len(nlp.Defaults.stop_words),
)
print(
    "'not' is a stopword in sklearn:",
    "not" in ENGLISH_STOP_WORDS,
    "| in spaCy:",
    "not" in nlp.Defaults.stop_words,
)

kept = []
for token in nlp(ticket):
    if not token.is_stop and not token.is_punct:
        kept.append(token.text)
print("after default removal:", kept)
assert "not" not in kept

sklearn list has 318 words | spaCy list has 326
'not' is a stopword in sklearn: True | in spaCy: True
after default removal: ['able', 'refund', 'app', 'working']


**Reading the output.** The lists differ in size, and both contain "not". After default removal the ticket reads `able get refund app working` — the two "not"s are gone, and with them the fact that nothing works.

**Keep the words that carry polarity.** Take the default list and remove the negations before using it.

In [3]:
keep = {"not", "no", "never", "nor", "cannot", "n't"}
custom_stopwords = set(nlp.Defaults.stop_words) - keep

kept = []
for token in nlp(ticket):
    if token.text.lower() not in custom_stopwords and not token.is_punct:
        kept.append(token.text)
print("with negations kept: ", kept)
assert kept.count("not") == 2

with negations kept:  ['not', 'able', 'refund', 'app', 'not', 'working']


**The rule to remember.** Stopword removal is for counting and keyword search. Start from a standard list, remove the negations, and never apply it to text an LLM or embedding model will read.

| Use it when | Don't when | Instead use |
|---|---|---|
| word clouds, TF-IDF/BM25 features, topic counts | sentiment, intent, anything where "not" matters; any neural model input | keep all words; let TF-IDF weighting down-rank common ones |

**Watch out**
- Domain words can be stopwords too: in a support corpus "ticket" and "please" carry nothing. Add them.
- TF-IDF already gives common words low weight; removal is often unnecessary there.
- Lists are English; every language has its own, and spaCy ships them per model.